# Viveka — Sealed Eval Inference: Qwen-2.5-7B

Standalone inference notebook for the trained Qwen-2.5-7B LoRA. Runs four `inference.py` passes (base T1+T2, base T3+T4, trained T1+T2, trained T3+T4) and pushes everything to HF Hub at **`ddevMhrn/Qwen2.5-7B-Viveka`**.

Built for the recovery path: training was done in an earlier Kaggle session, you downloaded `qwen25_7b_v1/` as a zip, and now want to run sealed eval without re-training.

**Gating note.** Qwen-2.5-7B is open access (no license gate), but inference uses Unsloth's pre-quantized 4-bit mirror `unsloth/qwen2.5-7b-instruct-unsloth-bnb-4bit` for memory reasons — `inference.py`'s `FrozenQwenPolicy` loads via plain `transformers.AutoModelForCausalLM` without a `BitsAndBytesConfig`, so the BF16 7B (~14 GB) would crowd T4's 14.56 GB ceiling. Same model Unsloth used during training; model card still credits the canonical Qwen id.

## Prereqs (do these BEFORE running)

1. **Notebook Settings:** Accelerator = `GPU T4 x2`, Internet = `On`, Persistence = `Files only`
2. **Add-ons → Secrets:** add `HF_TOKEN` (HuggingFace **write** token — needed to push to `ddevMhrn/Qwen2.5-7B-Viveka`)
3. **Add-ons → Datasets:** upload your zipped run folder as a new Kaggle Dataset:
   - Datasets → New Dataset → upload `qwen25_7b_v1.zip` (or whatever you named it)
   - Give it a memorable title like `viveka-qwen7b-run`
   - Attach to this notebook via **Add Input → + Add Input → select the dataset**

## Time budget

| Phase | Scenarios (at `--per-tier 5`) | Per-scenario | Total |
|---|---|---|---|
| Base T1+T2 | 10 | ~10-15 min | ~1.5-2.5 h |
| Base T3+T4 | 10 | ~10-15 min | ~1.5-2.5 h |
| Trained T1+T2 | 10 | ~5-8 min | ~1-1.5 h |
| Trained T3+T4 | 10 | ~5-8 min | ~1-1.5 h |
| **Total** | 40 | | **~5-8 h** |

`--per-tier 5` matches the published `llama1b_*` / `llama3b_*` convention. Bump it to 25 for full 68-scenario coverage if you have a longer Kaggle session and can afford ~12-16 h.


In [ ]:
# Step 1: GPU check + clone repo + set HF_TOKEN ─────────────────────
import os
from kaggle_secrets import UserSecretsClient

# Per-model config (hardcoded for this notebook — see inference_*_kaggle.ipynb
# for the other model). All downstream cells read these values.
BASE_MODEL  = "unsloth/qwen2.5-7b-instruct-unsloth-bnb-4bit"
RUN_NAME    = "qwen25_7b_v1"
ZIP_PATTERN = "qwen25_7b"
REPO_ID     = "ddevMhrn/Qwen2.5-7B-Viveka"
LOG_PREFIX  = "qwen7b"

print(f"Model:    {BASE_MODEL}")
print(f"Run dir:  {RUN_NAME}")
print(f"Push to:  {REPO_ID}")

# Move to known-good cwd before any rm/clone
os.chdir("/")
os.chdir("/kaggle/working")
%cd /kaggle/working

!nvidia-smi | head -20

# Clone from the HF Space (public; no GitHub token needed).
!rm -rf /kaggle/working/viveka-env
!git lfs install --skip-repo 2>/dev/null || true
!git clone https://huggingface.co/spaces/ddevMhrn/viveka-env /kaggle/working/viveka-env
%cd /kaggle/working/viveka-env

# HF_TOKEN with WRITE scope (needed for the final push)
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HUGGING_FACE_HUB_TOKEN"] = os.environ["HF_TOKEN"]
print("\nHF_TOKEN set:", bool(os.environ.get("HF_TOKEN")))


## Step 2: Lean inference install

Skips `trl`, `unsloth`, `mergekit`, `weave` — those are training-only. Inference needs only:
- `openenv-core` + `fastmcp` + `mcp` (env)
- `transformers` + `peft` (model + LoRA loading)
- `bitsandbytes` (4-bit weight loading)
- `huggingface_hub[cli]` (final push)

~3-4 min install vs ~8-10 min for the full training install.


In [ ]:
# Step 2: Install inference deps (lean — no training packages) ──────

# Project (main deps only, no [train] extras)
!pip install -q -e "."

# Pin openenv-core / fastmcp pair that works (same as training)
!pip install --upgrade --force-reinstall --no-deps "openenv-core==0.2.2" "fastmcp==3.1.1"

# mcp (Kaggle preinstalled is too old)
!pip install -q -U "mcp"

# openenv-core transitive
!pip install -q -U "uncalled-for"

# Inference essentials
!pip install -q -U "transformers>=4.40.0" "peft>=0.12" "bitsandbytes" "accelerate>=0.30.0"

# For HF push at the end
!pip install -q -U "huggingface_hub[cli]"

# Remove torchao: Kaggle preinstalls torchao 0.10.0, but peft >= 0.14 calls
# is_torchao_available() which RAISES ImportError when an installed torchao
# is older than 0.16.0 (instead of returning False cleanly). The raise bubbles
# up through PeftModel.from_pretrained() during LoRA loading and aborts
# inference. We don't use torchao for anything — bnb-4bit is the quant path —
# so removing it makes the dispatcher skip cleanly. See peft/import_utils.py.
!pip uninstall -y torchao 2>&1 | tail -2

print("\n=== installed versions ===")
!pip show openenv-core fastmcp mcp transformers peft bitsandbytes huggingface-hub 2>&1 | grep -E "^(Name|Version)" 


In [ ]:
# Step 3: Verify inference-side imports ─────────────────────────────
# No trl, no unsloth, no stubs needed.
import importlib, sys

def test(module):
    try:
        importlib.import_module(module)
        print(f"\u2705 {module}")
        return True
    except Exception as e:
        print(f"\u274c {module}: {type(e).__name__}: {e}")
        return False

ok = True
ok &= test("transformers")
ok &= test("peft")
ok &= test("bitsandbytes")
ok &= test("openenv.core.rubrics.trajectory")
ok &= test("openenv.core.env_server.types")

try:
    sys.path.insert(0, "/kaggle/working/viveka-env")
    from viveka.server.environment import VivekaEnvironment  # noqa: F401
    print("\u2705 viveka.server.environment.VivekaEnvironment")
except Exception as e:
    print(f"\u274c viveka env: {type(e).__name__}: {e}"); ok = False

try:
    from inference import FrozenQwenPolicy  # noqa: F401
    print("\u2705 inference.FrozenQwenPolicy")
except Exception as e:
    print(f"\u274c inference policy: {type(e).__name__}: {e}"); ok = False

print(f"\n{'\u2705 ALL CLEAN — proceed' if ok else '\u274c FIX BEFORE PROCEEDING'}")


## Step 4: Locate the trained LoRA (handles all input shapes)

Kaggle's behavior when you upload a `.zip` as a Dataset varies:
- **Auto-extracted**: `/kaggle/input/<dataset>/` contains the extracted file tree directly (no `.zip` file present)
- **Stored as zip**: `/kaggle/input/<dataset>/<filename>.zip` is the raw zip we need to extract ourselves

The cell below handles both. It searches recursively for `lora/adapter_config.json` in `/kaggle/input/` first (covers auto-extracted case). If nothing's there, it falls back to extracting a `.zip` matching `'qwen25_7b'` (or any single `.zip` available) to a **staging directory** (`/kaggle/working/_staging/`) — not `runs/` — so the freshly-cloned `viveka-env/` isn't clobbered. Then it copies just the LoRA folder and `training_log.jsonl` to the canonical `/kaggle/working/runs/qwen25_7b_v1/`. Checkpoints are skipped (inference doesn't need them; saves ~500 MB).


In [ ]:
# Step 4: Locate LoRA from input (handles extracted + zipped cases) ─
import zipfile, shutil
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")
STAGING    = Path("/kaggle/working/_staging")
TARGET_RUN = Path("/kaggle/working/runs") / RUN_NAME
STAGING.mkdir(parents=True, exist_ok=True)
TARGET_RUN.mkdir(parents=True, exist_ok=True)

# --- Step A: scan input for an already-extracted lora/ folder ----------
print(f"Scanning {INPUT_ROOT} for an adapter_config.json ...")
extracted_configs = list(INPUT_ROOT.rglob("lora/adapter_config.json"))
print(f"  found {len(extracted_configs)} adapter configs in /kaggle/input/")
for p in extracted_configs[:5]:
    print(f"    {p}")

src_lora_dir = None
src_run_dir  = None

if extracted_configs:
    # Pick the one most likely to be ours: prefer paths containing RUN_NAME
    preferred = [p for p in extracted_configs if RUN_NAME in str(p)]
    chosen = (preferred or extracted_configs)[0]
    src_lora_dir = chosen.parent          # .../lora
    src_run_dir  = chosen.parent.parent   # parent of lora/ — may or may not be named RUN_NAME
    print(f"\nUsing extracted LoRA at {src_lora_dir}")
else:
    # --- Step B: fall back to extracting a .zip ------------------------
    zips = list(INPUT_ROOT.rglob("*.zip"))
    print(f"\nNo extracted LoRA found. Looking for zips in {INPUT_ROOT} ...")
    for z in zips:
        print(f"  {z}  ({z.stat().st_size // 1024 // 1024} MB)")

    matching = [z for z in zips if ZIP_PATTERN in str(z)]
    if not matching and len(zips) == 1:
        matching = zips
        print(f"\nNo zip matched pattern {ZIP_PATTERN!r}; using the single available zip")
    if not matching:
        raise FileNotFoundError(
            f"Neither lora/ nor matching .zip found in /kaggle/input/. "
            f"Verify you attached the Kaggle Dataset with the {RUN_NAME} output."
        )

    zip_path = matching[0]
    print(f"\nExtracting {zip_path} → {STAGING} ...")
    with zipfile.ZipFile(zip_path, "r") as zf:
        # Selective extraction: pull only members we actually need to inference
        # (lora/, training_log.jsonl). Skips checkpoints — saves disk + time.
        needed = [n for n in zf.namelist()
                  if "/lora/" in n or n.endswith("/lora") or n.endswith("training_log.jsonl")]
        # If selective filter caught nothing, fall back to extractall (zip layout unfamiliar)
        if not needed:
            print("  (selective filter caught nothing — extracting full zip)")
            zf.extractall(STAGING)
        else:
            print(f"  selectively extracting {len(needed)} members (skipping checkpoints)")
            zf.extractall(STAGING, members=needed)

    # Find lora/ in the staging output — prefer RUN_NAME if multiple exist
    # (e.g., zip may include both runs/qwen25_7b_v1/lora and smoke/lora).
    staged_configs = list(STAGING.rglob("lora/adapter_config.json"))
    if not staged_configs:
        raise FileNotFoundError(
            f"Extracted {zip_path} but found no lora/adapter_config.json inside."
        )
    preferred_staged = [p for p in staged_configs if RUN_NAME in str(p)]
    chosen = (preferred_staged or staged_configs)[0]
    if len(staged_configs) > 1 and not preferred_staged:
        print(f"\u26a0\ufe0f  {len(staged_configs)} lora/ folders found, none match RUN_NAME={RUN_NAME!r}; using {chosen}")
    src_lora_dir = chosen.parent
    src_run_dir  = chosen.parent.parent
    print(f"\nUsing extracted LoRA at {src_lora_dir}")

# --- Step C: copy LoRA + training_log to canonical /kaggle/working/runs/<RUN_NAME>/
target_lora = TARGET_RUN / "lora"
if target_lora.exists():
    print(f"\nRemoving existing {target_lora}")
    shutil.rmtree(target_lora)
print(f"Copying {src_lora_dir} → {target_lora}")
shutil.copytree(src_lora_dir, target_lora)

src_log = src_run_dir / "training_log.jsonl"
if src_log.exists():
    shutil.copy(src_log, TARGET_RUN / "training_log.jsonl")
    print(f"Copied training_log.jsonl")
else:
    print(f"\u26a0\ufe0f  training_log.jsonl not found at {src_log} (reward curve will be skipped)")

print(f"\n\u2705 ready. canonical run dir = {TARGET_RUN}")
!ls -la $TARGET_RUN/ $TARGET_RUN/lora/ 2>&1 | head -25


In [ ]:
# Step 5: Set canonical paths for downstream cells ──────────────────
from pathlib import Path

RUN_DIR = Path("/kaggle/working/runs") / RUN_NAME
LORA_DIR = RUN_DIR / "lora"
LOG_FILE = RUN_DIR / "training_log.jsonl"

assert LORA_DIR.exists() and (LORA_DIR / "adapter_config.json").exists(), \
    f"missing LoRA at {LORA_DIR}/adapter_config.json — Step 4 didn't complete cleanly"

# String forms for IPython $var expansion in subsequent !-cells
RUN_DIR_STR  = str(RUN_DIR)
LORA_DIR_STR = str(LORA_DIR)

print(f"RUN_DIR_STR  = {RUN_DIR_STR}")
print(f"LORA_DIR_STR = {LORA_DIR_STR}")
print(f"LOG_FILE     = {LOG_FILE} (exists={LOG_FILE.exists()})")


In [ ]:
# Step 6: Reward curve (optional, ~30 sec) ──────────────────────────
# Regenerates the training reward curve from training_log.jsonl. Output PNG
# ends up in RUN_DIR so it ships with the eval logs in the HF push.

if LOG_FILE.exists():
    out_png = f"{RUN_DIR_STR}/reward_curve.png"
    !cd /kaggle/working/viveka-env && python eval/reward_curve.py \
        --training-log $RUN_DIR_STR/training_log.jsonl \
        --baseline-json eval/results/baseline_random.json \
        --output-png $out_png \
        --smooth-window 10 \
        --title "GRPO Training — Qwen-2.5-7B (200 episodes)" 2>&1 | tail -5
else:
    print(f"\u26a0\ufe0f  no training_log.jsonl in {RUN_DIR_STR} — skipping reward curve")

!ls -la $RUN_DIR_STR/ | head -15


## Sealed eval — base vs trained, T1+T2 then T3+T4

Same four-pass pattern as the published `eval/results/llama1b_*` / `llama3b_*` logs. `--per-tier 5` matches the published convention; bump to 25 if you have time.

**T3+T4 trained log is the showcase number** — its T4 row tells you how many `must_not_execute` traps fired.


In [ ]:
# Step 7: Inference — FROZEN base on T1+T2 ─────────────────────────
# Base model (no LoRA). Untrained models typically don't terminate — every
# scenario hits MAX_STEPS=30 — so this is the slowest of the four passes.

# Pre-build the full output paths in Python so IPython's $var expansion
# doesn't get confused by $LOG_PREFIX_base_t12 (greedy underscore matching).
out_json = f"{RUN_DIR_STR}/{LOG_PREFIX}_base_t12.json"
out_log  = f"{RUN_DIR_STR}/{LOG_PREFIX}_base_t12.log"

!cd /kaggle/working/viveka-env && CUDA_VISIBLE_DEVICES=0 python inference.py \
    --policy qwen \
    --model $BASE_MODEL \
    --tier-mix 1,2 \
    --per-tier 5 \
    --output-json $out_json \
    2>&1 | tee $out_log


In [ ]:
# Step 8: Inference — FROZEN base on T3+T4 ─────────────────────────
out_json = f"{RUN_DIR_STR}/{LOG_PREFIX}_base_t34.json"
out_log  = f"{RUN_DIR_STR}/{LOG_PREFIX}_base_t34.log"

!cd /kaggle/working/viveka-env && CUDA_VISIBLE_DEVICES=0 python inference.py \
    --policy qwen \
    --model $BASE_MODEL \
    --tier-mix 3,4 \
    --per-tier 5 \
    --output-json $out_json \
    2>&1 | tee $out_log


In [ ]:
# Step 9: Inference — TRAINED (base + LoRA) on T1+T2 ───────────────
out_json = f"{RUN_DIR_STR}/{LOG_PREFIX}_train_t12.json"
out_log  = f"{RUN_DIR_STR}/{LOG_PREFIX}_train_t12.log"

!cd /kaggle/working/viveka-env && CUDA_VISIBLE_DEVICES=0 python inference.py \
    --policy qwen \
    --model $BASE_MODEL \
    --adapter $LORA_DIR_STR \
    --tier-mix 1,2 \
    --per-tier 5 \
    --output-json $out_json \
    2>&1 | tee $out_log


In [ ]:
# Step 10: Inference — TRAINED on T3+T4 (the showcase tier) ────────
out_json = f"{RUN_DIR_STR}/{LOG_PREFIX}_train_t34.json"
out_log  = f"{RUN_DIR_STR}/{LOG_PREFIX}_train_t34.log"

!cd /kaggle/working/viveka-env && CUDA_VISIBLE_DEVICES=0 python inference.py \
    --policy qwen \
    --model $BASE_MODEL \
    --adapter $LORA_DIR_STR \
    --tier-mix 3,4 \
    --per-tier 5 \
    --output-json $out_json \
    2>&1 | tee $out_log

print("\n=== compare base vs trained (all 4 summaries) ===")
log_glob = f"{RUN_DIR_STR}/{LOG_PREFIX}_*.log"
!grep -A 6 "SUMMARY" $log_glob


## Push everything to HF Hub: `ddevMhrn/Qwen2.5-7B-Viveka`

Bundles `lora/` + `training_log.jsonl` + `reward_curve.png` + all 4 eval logs (`*.log` and `*.json`) into one folder, writes a model card, then uploads. Idempotent — safe to re-run.


In [ ]:
# Step 11: Push to HF Hub ───────────────────────────────────────────
import os, shutil
from pathlib import Path
from huggingface_hub import HfApi, create_repo

assert LORA_DIR.exists(), f"missing LoRA dir: {LORA_DIR}"

# Bundle every artifact into the LoRA folder so one upload ships the lot
for fname in [
    "training_log.jsonl",
    "reward_curve.png",
    f"{LOG_PREFIX}_base_t12.log",   f"{LOG_PREFIX}_base_t12.json",
    f"{LOG_PREFIX}_base_t34.log",   f"{LOG_PREFIX}_base_t34.json",
    f"{LOG_PREFIX}_train_t12.log",  f"{LOG_PREFIX}_train_t12.json",
    f"{LOG_PREFIX}_train_t34.log",  f"{LOG_PREFIX}_train_t34.json",
]:
    src = RUN_DIR / fname
    if src.exists():
        shutil.copy(src, LORA_DIR / fname)
        print(f"  bundled {fname}")

# Model card. `base_model` points at the canonical id even where eval used
# Unsloth's open mirror — the LoRA was trained on the canonical weights.
card = """---
library_name: peft
base_model: Qwen/Qwen2.5-7B-Instruct
license: apache-2.0
tags:
  - viveka
  - grpo
  - reversibility
  - calibrated-confidence
  - indic-dpi
  - openenv
---

# Qwen2.5-7B-Viveka

LoRA adapter trained on the [Viveka OpenEnv](https://huggingface.co/spaces/ddevMhrn/viveka-env) with TRL GRPO + Unsloth 4-bit QLoRA. Six-component deterministic reward over mocked Indian DPI services (UPI, DigiLocker, IRCTC, Banking, Telecom). 200 episodes, tier mix 1:0.4 / 2:0.4 / 4:0.2.

**Base model:** `Qwen/Qwen2.5-7B-Instruct`

**Notes:** Same train.py config as the v6 Qwen-1.5B run. No OOM mitigations needed on T4 x2.

See [github.com/DevMhrn/viveka-env](https://github.com/DevMhrn/viveka-env) for the env, reward design, and eval harness.
"""
(LORA_DIR / "README.md").write_text(card)
print("  wrote README.md")

# Create repo + upload
create_repo(REPO_ID, repo_type="model", exist_ok=True, private=False, token=os.environ["HF_TOKEN"])
HfApi().upload_folder(
    folder_path=str(LORA_DIR),
    repo_id=REPO_ID,
    repo_type="model",
    token=os.environ["HF_TOKEN"],
    commit_message="add sealed-eval (base + trained, T1+T2 + T3+T4) and reward curve",
)
print(f"\n\u2705 pushed to https://huggingface.co/{REPO_ID}")
